In [ ]:
# Conta que rende a taxa livre de risco (RF) diariamente
class CarryBroker(bt.brokers.BackBroker):
    params = dict(carry=RF/100)

    def next(self):
        daily_carry = (1 + self.p.carry) ** (1/252) - 1
        self.cash += self.cash * daily_carry
        super().next()


# Função que faz a simulação com o report
def simulate(strategy, data):

    cerebro = bt.Cerebro(stdstats=True)  # Cria o Engine de simulação
    cerebro.broker = CarryBroker(carry=RF/100)
    cerebro.adddata(data)  # Adiciona os dados
    cerebro.addstrategy(strategy)  # Adiciona a estratégia

    CUSTO_EMOLUMENTOS = (0.005 + 0.0275) / 100  # taxa de 0.005% + 0.0275% (emolumentos)

    # Ajustando parâmetros de carteira
    cerebro.broker.setcash(1_000_000)
    cerebro.broker.setcommission(commission=CUSTO_EMOLUMENTOS)
    cerebro.broker.set_coc(True)  # cheat on close - para comprar no mesmo dia

    # Performance & Risk Metrics
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.TimeReturn, timeframe=bt.TimeFrame.NoTimeFrame, _name='timereturn')
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', timeframe=bt.TimeFrame.Days, riskfreerate=RF/100, annualize=True)
    cerebro.addanalyzer(bt.analyzers.SQN, _name='sqn')  # System Quality Number
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Calmar, _name='calmar')
    cerebro.addanalyzer(bt.analyzers.VWR, _name='vwr', timeframe=bt.TimeFrame.Days)  # Variability-Weighted Return

    # Trade
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trade_analyzer')
    cerebro.addanalyzer(bt.analyzers.Transactions, _name='transactions')
    cerebro.addanalyzer(bt.analyzers.PositionsValue, _name='positions')

    # Drawdown
    cerebro.addanalyzer(bt.analyzers.TimeDrawDown, _name='timedrawdown')
    cerebro.addanalyzer(bt.analyzers.GrossLeverage, _name='leverage')

    # Portfolio Analysis
    cerebro.addanalyzer(bt.analyzers.PyFolio, _name='pyfolio')

    # Run the strategy
    results = cerebro.run()
    strat = results[0]

    print("=" * 60)
    print("PERFORMANCE METRICS SUMMARY")
    print("=" * 60)

    # Basic Returns
    returns_analyzer = strat.analyzers.returns.get_analysis()
    print(f"Total Return: {returns_analyzer['rtot']:.2%}")
    print(f"Average Return: {returns_analyzer['ravg']:.4%}")
    print(f"Compound Annual Growth Rate: {returns_analyzer['rnorm100']:.2f}%")

    # Sharpe Ratio
    sharpe = strat.analyzers.sharpe.get_analysis()
    print(f"Sharpe Ratio: {sharpe.get('sharperatio'):.4f}" if sharpe.get('sharperatio') is not None else "Sharpe Ratio: N/A")

    # System Quality Number
    sqn = strat.analyzers.sqn.get_analysis()
    print(f"System Quality Number (SQN): {sqn.get('sqn'):.4f}" if sqn.get('sqn') is not None else "SQN: N/A")

    # DrawDown Analysis
    dd = strat.analyzers.drawdown.get_analysis()
    print(f"Max Drawdown: {dd.max.drawdown:.2f}%")
    print(f"Max Drawdown Duration: {dd.max.len} periods")
    print(f"Max Drawdown Money: ${dd.max.moneydown:,.2f}")

    # Calmar Ratio
    calmar = strat.analyzers.calmar.get_analysis()
    print(f"Calmar Ratio: {calmar.get('calmar'):.4f}" if calmar.get('calmar') is not None else "Calmar Ratio: N/A")

    # VWR - Variability-Weighted Return
    vwr = strat.analyzers.vwr.get_analysis()
    print(f"VWR: {vwr.get('vwr'):.4f}" if vwr.get('vwr') is not None else "VWR: N/A")

    print("\n" + "=" * 60)
    print("TRADE ANALYSIS")
    print("=" * 60)

    trade_analysis = strat.analyzers.trade_analyzer.get_analysis()
    total_trades = trade_analysis.total.get('total', 0)
    print(f"Total Trades: {total_trades}")

    if total_trades > 0:
        print(f"Total Open Trades: {trade_analysis.total.get('open', 0)}")
        print(f"Total Closed Trades: {trade_analysis.total.get('closed', 0)}")

        won = trade_analysis.get('won', {})
        lost = trade_analysis.get('lost', {})

        print(f"\nWinning Trades: {won.get('total', 0)}")
        print(f"Losing Trades: {lost.get('total', 0)}")
        print(f"Win Rate: {(won.get('total', 0) / total_trades) * 100:.1f}%")

        if won.get('total', 0) > 0:
            print(f"Average Win: ${won.pnl.get('average', 0):,.2f}")
            print(f"Max Win: ${won.pnl.get('max', 0):,.2f}")

        if lost.get('total', 0) > 0:
            print(f"Average Loss: ${lost.pnl.get('average', 0):,.2f}")
            print(f"Max Loss: ${lost.pnl.get('max', 0):,.2f}")

        if 'pnl' in trade_analysis:
            print(f"Average Trade: ${trade_analysis.pnl.get('net', {}).get('average', 0):,.2f}")
        if 'len' in trade_analysis:
            print(f"Average Trade Length: {trade_analysis.len.get('average', 0):.1f} periods")

        # Profit Factor
        gross_profit = won.pnl.get('total', 0) if won.get('total', 0) > 0 else 0
        gross_loss = abs(lost.pnl.get('total', 0)) if lost.get('total', 0) > 0 else 0
        if gross_loss > 0:
            print(f"Profit Factor: {gross_profit / gross_loss:.2f}")

    print("\n" + "=" * 60)
    print("PORTFOLIO VALUE")
    print("=" * 60)
    print(f"Starting Portfolio Value: $1,000,000.00")
    print(f"Final Portfolio Value: ${cerebro.broker.getvalue():,.2f}")
    print(f"Total P&L: ${cerebro.broker.getvalue() - 1_000_000:,.2f}")

    # Pyfolio
    pyfolio_analyzer = strat.analyzers.pyfolio
    returns, positions, transactions, gross_lev = pyfolio_analyzer.get_pf_items()
    print(f"\nReturns Series Length: {len(returns)}")
    print(f"Positions Tracked: {len(positions.columns) if hasattr(positions, 'columns') else 0}")
    print(f"Transactions Recorded: {len(transactions)}")

    figs = cerebro.plot(style='candlestick', volume=False, iplot=False, numfigs=1, figsize=(20, 12), dpi=120)
    plt.show()

    return strat


# Roda a simulação para cada ação selecionada
for ticker in tickers:
    print(f"\n{'#' * 60}\n{ticker}\n{'#' * 60}")
    data = bt.feeds.PandasData(dataname=prices[ticker])
    simulate(RSIMeanReversion, data)